In [1]:
from itertools import cycle
import numpy as np
import matplotlib.pyplot as plt
import scipy
import pandas as pd
import xgboost as xgb
from sklearn import linear_model
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV, RepeatedKFold
from sklearn.preprocessing import StandardScaler 
from sklearn.datasets import make_regression
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, roc_curve, auc
from sklearn.metrics import make_scorer
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from string import ascii_uppercase
from geopy.distance import geodesic
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.multioutput import MultiOutputRegressor
from sklearn import linear_model
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
import numpy as np
from xgboost import XGBRegressor
from tqdm import tqdm
import time
from scipy.stats import uniform
from sklearn.pipeline import Pipeline

from sklearn.linear_model import Lasso, LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error,r2_score,mean_squared_error

In [2]:
import sys
import os

# Get the absolute path to your src folder
module_path = os.path.abspath(os.path.join('..', 'src'))

if module_path not in sys.path:
    sys.path.append(module_path)

from cleaning_and_helpers import plot_test_preds


In [3]:
np.random.seed(1298)

In [4]:
SI_X = np.load('../../pollenGeolocation_saved/SI_X_tax.npy')
SI_y = np.load('../../pollenGeolocation_saved/SI_y.npy')

FFAR_X = np.load('../../pollenGeolocation_saved/FFAR_X_tax.npy')
FFAR_y = np.load('../../pollenGeolocation_saved/FFAR_y.npy')

NCASI_X = np.load('../../pollenGeolocation_saved/NCASI_X_tax.npy')
NCASI_y = np.load('../../pollenGeolocation_saved/NCASI_y.npy')

In [5]:
# Train test split for each project
seed = 1298
test_size = 0.20

# --- Function to split each project ---
def split_project(X, y, test_size, random_state):
    return train_test_split(X, y, test_size=test_size, random_state=random_state)

# --- Split each project ---
SI_X_train, SI_X_test, SI_y_train, SI_y_test = split_project(SI_X, SI_y, test_size, seed)
NCASI_X_train, NCASI_X_test, NCASI_y_train, NCASI_y_test = split_project(NCASI_X, NCASI_y, test_size, seed)
FFAR_X_train, FFAR_X_test, FFAR_y_train, FFAR_y_test = split_project(FFAR_X, FFAR_y, test_size, seed)

# --- Concatenate train and test splits from all projects ---
X_train = np.concatenate([SI_X_train, NCASI_X_train, FFAR_X_train], axis=0)
y_train = np.concatenate([SI_y_train, NCASI_y_train, FFAR_y_train], axis=0)

X_test = np.concatenate([SI_X_test, NCASI_X_test, FFAR_X_test], axis=0)
y_test = np.concatenate([SI_y_test, NCASI_y_test, FFAR_y_test], axis=0)

# --- Standardize X and y using training data only ---
sc_X = StandardScaler()
sc_y = StandardScaler()

X_train = sc_X.fit_transform(X_train)
y_train = sc_y.fit_transform(y_train)

X_test = sc_X.transform(X_test)
y_test = sc_y.transform(y_test)


In [6]:
# Project tracking 

# Add project IDs
SI_project_train = np.full(len(SI_X_train), 'SI')
SI_project_test  = np.full(len(SI_X_test), 'SI')

NCASI_project_train = np.full(len(NCASI_X_train), 'NCASI')
NCASI_project_test  = np.full(len(NCASI_X_test), 'NCASI')

FFAR_project_train = np.full(len(FFAR_X_train), 'FFAR')
FFAR_project_test  = np.full(len(FFAR_X_test), 'FFAR')

# Concatenate project labels
project_train = np.concatenate([SI_project_train, NCASI_project_train, FFAR_project_train], axis=0)
project_test  = np.concatenate([SI_project_test, NCASI_project_test, FFAR_project_test], axis=0)


In [7]:
## models
untuned_models = {
    "MultiTaskLasso": lambda: linear_model.MultiTaskLasso(random_state=seed),
    "SVR": lambda: MultiOutputRegressor(SVR()),
    "KNN": lambda: MultiOutputRegressor(KNeighborsRegressor()),
    "DecisionTree": lambda: MultiOutputRegressor(DecisionTreeRegressor(random_state=seed)),
    "RandomForest": lambda: MultiOutputRegressor(RandomForestRegressor(random_state=seed)),
    "XGBoost": lambda: MultiOutputRegressor(XGBRegressor(random_state=seed))
}

even_tuned_models = {
    "MultiTaskLasso": lambda: linear_model.MultiTaskLasso(alpha=0.1, 
                                                          random_state=seed),
    "SVR": lambda: MultiOutputRegressor(SVR(C=10,
                                            gamma='auto')),
    "KNN": lambda: MultiOutputRegressor(KNeighborsRegressor(n_neighbors=3)),
    "DecisionTree": lambda: MultiOutputRegressor(DecisionTreeRegressor(max_depth=15,
                                                                       random_state=seed)),
    "RandomForest": lambda: MultiOutputRegressor(RandomForestRegressor(max_depth=20,
                                                                       n_estimators=150,
                                                                       random_state=seed)),
    "XGBoost": lambda: MultiOutputRegressor(XGBRegressor(max_depth=6,
                                                         n_estimators=100,
                                                         random_state=seed))
}

best_tuned_models = {
    "MultiTaskLasso": lambda: linear_model.MultiTaskLasso(alpha=0.08,
                                                          random_state=seed),
    "SVR": lambda: MultiOutputRegressor(SVR(C=2.847162213648956,
                                            gamma='auto',
                                            kernel='rbf')),
    "KNN": lambda: MultiOutputRegressor(KNeighborsRegressor(weights='uniform',
                                                            p=2,
                                                            n_neighbors=3,
                                                            algorithm='auto')),
    "DecisionTree": lambda: MultiOutputRegressor(DecisionTreeRegressor(min_samples_split=10,
                                                                       min_samples_leaf=2,
                                                                       max_leaf_nodes=40,
                                                                       max_features=None,
                                                                       max_depth=18,
                                                                       random_state=seed)),
    "RandomForest": lambda: MultiOutputRegressor(RandomForestRegressor(n_estimators=200,
                                                                       min_samples_split=5,
                                                                       min_samples_leaf=1,
                                                                       max_features='log2',
                                                                       max_depth=None,
                                                                       random_state=seed)),
    "XGBoost": lambda: MultiOutputRegressor(XGBRegressor(subsample=0.8,
                                                         reg_lambda=1,
                                                         reg_alpha=1,
                                                         n_estimators=500,
                                                         min_child_weight=1,
                                                         max_depth=20,
                                                         learning_rate=0.3,
                                                         gamma=0,
                                                         colsample_bytree=0.8,
                                                         random_state=seed))
}

In [8]:
from sklearn.metrics import r2_score, mean_squared_error, median_absolute_error
from geopy.distance import geodesic
import numpy as np
import pandas as pd

def evaluate_model(name, model_class, X_train, y_train, X_test, y_test, project_test=None):
    """
    Initialize, fit, predict, and evaluate a model with default hyperparameters.
    """
    print(f"Evaluating {name}...")

    # If it’s a lambda or a class, calling it gets you a fresh model
    model = model_class()

    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    # Overall metrics
    r2 = r2_score(y_test, preds)
    rmse = mean_squared_error(y_test, preds, squared=False)
    mae = median_absolute_error(y_test, preds)
    distances = [geodesic(real, pred).kilometers for real, pred in zip(y_test, preds)]
    avg_dist = np.mean(distances)
    se_dist = np.std(distances)

    results = {
        "model": name,
        "r2": r2,
        "rmse": rmse,
        "mae": mae,
        "avg_km_error": avg_dist,
        "se_km_error": se_dist,
        "preds": preds
    }

    # ---- Compute per-project metrics ----
    if project_test is not None:
        df = pd.DataFrame({
            "y_true_lat": [y[0] for y in y_test],
            "y_true_lon": [y[1] for y in y_test],
            "y_pred_lat": [y[0] for y in preds],
            "y_pred_lon": [y[1] for y in preds],
            "project": project_test
        })

        project_metrics = []

        for project_id, group in df.groupby("project"):
            y_true_proj = group[["y_true_lat", "y_true_lon"]].values
            y_pred_proj = group[["y_pred_lat", "y_pred_lon"]].values

            r2_p = r2_score(y_true_proj, y_pred_proj)
            rmse_p = mean_squared_error(y_true_proj, y_pred_proj, squared=False)
            mae_p = median_absolute_error(y_true_proj, y_pred_proj)
            distances_p = [geodesic(real, pred).kilometers for real, pred in zip(y_true_proj, y_pred_proj)]
            avg_dist_p = np.mean(distances_p)
            se_dist_p = np.std(distances_p)

            project_metrics.append({
                "project": project_id,
                "r2": r2_p,
                "rmse": rmse_p,
                "mae": mae_p,
                "avg_km_error": avg_dist_p,
                "se_km_error": se_dist_p
            })

        results["per_project"] = project_metrics

    return results


In [9]:
# Final results storage
all_results = []

# Your dict of dicts
model_groups = {
    "untuned": untuned_models,
    "even_tuned": even_tuned_models,
    "best_tuned": best_tuned_models
}


for group_name, models in model_groups.items():
    print(f"--- Running models for group: {group_name} ---")

    for model_name, model_class in models.items():
        result = evaluate_model(
            name=model_name,
            model_class=model_class,
            X_train=X_train,
            y_train=y_train,
            X_test=X_test,
            y_test=y_test,
            project_test=project_test  # <-- so you get per-project stats
        )

        # Add tracking info
        result["group"] = group_name  # untuned/even_tuned/best_tuned

        all_results.append(result)


--- Running models for group: untuned ---
Evaluating MultiTaskLasso...
Evaluating SVR...
Evaluating KNN...
Evaluating DecisionTree...
Evaluating RandomForest...
Evaluating XGBoost...
--- Running models for group: even_tuned ---
Evaluating MultiTaskLasso...
Evaluating SVR...
Evaluating KNN...
Evaluating DecisionTree...
Evaluating RandomForest...
Evaluating XGBoost...
--- Running models for group: best_tuned ---
Evaluating MultiTaskLasso...
Evaluating SVR...
Evaluating KNN...
Evaluating DecisionTree...
Evaluating RandomForest...
Evaluating XGBoost...


In [10]:
# flatten for CSV or DataFrame
flat_rows = []
for result in all_results:
    for proj_metric in result.get("per_project", []):
        flat_rows.append({
            "group": result["group"],
            "model": result["model"],
            "project": proj_metric["project"],
            "r2": proj_metric["r2"],
            "rmse": proj_metric["rmse"],
            "mae": proj_metric["mae"],
            "avg_km_error": proj_metric["avg_km_error"],
            "se_km_error": proj_metric["se_km_error"]
        })

df_per_project = pd.DataFrame(flat_rows)

# Sort by project, then group, then model (or any order you prefer)
df_per_project = df_per_project.sort_values(
    by=["project", "group", "model"]
).reset_index(drop=True)

df_per_project



,group,model,project,r2,rmse,mae,avg_km_error,se_km_error
0,best_tuned,DecisionTree,FFAR,-17.583069,0.094108,0.022694,5.181678,17.885207
1,best_tuned,KNN,FFAR,0.186334,0.022187,0.012705,2.832407,2.122572
2,best_tuned,MultiTaskLasso,FFAR,-17.335025,0.096552,0.063492,12.550981,8.855195
3,best_tuned,RandomForest,FFAR,-9.249686,0.069828,0.014000,4.514994,10.348932
4,best_tuned,SVR,FFAR,-27.049155,0.123654,0.059725,13.804068,13.698285
5,best_tuned,XGBoost,FFAR,-2.863193,0.046459,0.015551,4.078016,7.248495
6,even_tuned,DecisionTree,FFAR,-24.962235,0.108084,0.013012,4.849516,21.535121
7,even_tuned,KNN,FFAR,0.186334,0.022187,0.012705,2.832407,2.122572
8,even_tuned,MultiTaskLasso,FFAR,-19.617101,0.100464,0.075054,13.989771,8.015499
9,even_tuned,RandomForest,FFAR,-4.166484,0.051253,0.011541,3.415308,9.148394


In [11]:
flat_rows = []

for result in all_results:
    # Add the overall metrics
    flat_rows.append({
        "group": result["group"],
        "model": result["model"],
        "project": "ALL",  # special label for full dataset
        "r2": result["r2"],
        "rmse": result["rmse"],
        "mae": result["mae"],
        "avg_km_error": result["avg_km_error"],
        "se_km_error": result["se_km_error"]
    })

    # Then add the per-project metrics
    for proj_metric in result.get("per_project", []):
        flat_rows.append({
            "group": result["group"],
            "model": result["model"],
            "project": proj_metric["project"],
            "r2": proj_metric["r2"],
            "rmse": proj_metric["rmse"],
            "mae": proj_metric["mae"],
            "avg_km_error": proj_metric["avg_km_error"],
            "se_km_error": proj_metric["se_km_error"]
        })

# Turn into DataFrame and order by project first, so ALL comes first if you like
df_per_project = pd.DataFrame(flat_rows).sort_values(
    by=["project", "model", "group"]
).reset_index(drop=True)

df_per_project


,group,model,project,r2,rmse,mae,avg_km_error,se_km_error
0,best_tuned,DecisionTree,ALL,0.866810,0.373581,0.023184,19.207829,55.389118
1,even_tuned,DecisionTree,ALL,0.851075,0.395547,0.014825,20.209882,58.707424
2,untuned,DecisionTree,ALL,0.846688,0.401282,0.014011,20.746558,59.504429
3,best_tuned,KNN,ALL,0.976115,0.152536,0.015578,10.163496,22.885707
4,even_tuned,KNN,ALL,0.976115,0.152536,0.015578,10.163496,22.885707
...,...,...,...,...,...,...,...,...
67,even_tuned,SVR,SI,-3.043036,0.629857,0.123302,59.965222,78.961911
68,untuned,SVR,SI,-3.760542,0.688018,0.187810,68.810109,83.455446
69,best_tuned,XGBoost,SI,-2.812418,0.628508,0.224633,59.890446,78.329558
70,even_tuned,XGBoost,SI,-6.109919,0.794409,0.193169,72.426920,104.559649


In [12]:
df_per_project.to_csv("../tables/taxonomic/tax_all_mod_compare.csv")

In [13]:
## WIP Diversity and evenness
# Keep the raw relative abundance values before scaling
X_train_raw = np.concatenate([SI_X_train, NCASI_X_train, FFAR_X_train], axis=0)
X_test_raw  = np.concatenate([SI_X_test, NCASI_X_test, FFAR_X_test], axis=0)

import pandas as pd

# Suppose you know the pollen DNA feature names
pollen_cols = [f"ASV_{i}" for i in range(X_train_raw.shape[1])]

# Combine train and test if you want diversity metrics for all samples
X_all_raw = np.concatenate([X_train_raw, X_test_raw], axis=0)
project_all = np.concatenate([project_train, project_test], axis=0)

df = pd.DataFrame(X_all_raw, columns=pollen_cols)
df['project'] = project_all


In [15]:
import numpy as np

def shannon_diversity(rel_abundances):
    """Compute Shannon diversity, ignoring zeros and NaNs"""
    rel_abundances = np.asarray(rel_abundances, dtype=float)
    rel_abundances = rel_abundances[np.isfinite(rel_abundances)]  # drop NaNs
    rel_abundances = rel_abundances[rel_abundances > 0]           # drop zeros
    if len(rel_abundances) == 0:
        return 0
    return -np.sum(rel_abundances * np.log(rel_abundances))

def pielou_evenness(shannon, richness):
    """Compute Pielou's evenness"""
    if richness > 1 and np.isfinite(shannon):
        return shannon / np.log(richness)
    else:
        return 0


In [16]:
diversity_results = []

for i, row in df.iterrows():
    rel_abund = row[pollen_cols].values
    richness = np.sum(rel_abund > 0)
    shannon  = shannon_diversity(rel_abund)
    evenness = pielou_evenness(shannon, richness)
    diversity_results.append({
        'project': row['project'],
        'richness': richness,
        'shannon': shannon,
        'evenness': evenness
    })

div_df = pd.DataFrame(diversity_results)


In [17]:
project_summary = div_df.groupby('project').agg(
    richness_mean=('richness', 'mean'),
    richness_sd=('richness', 'std'),
    shannon_mean=('shannon', 'mean'),
    shannon_sd=('shannon', 'std'),
    evenness_mean=('evenness', 'mean'),
    evenness_sd=('evenness', 'std')
).reset_index()

print(project_summary)


  project  richness_mean  richness_sd  shannon_mean  shannon_sd  \
0    FFAR       5.379457     4.070887      0.683937    0.577741   
1   NCASI       6.153409     4.053117      0.596505    0.518185   
2      SI       2.030702     1.100230      0.325354    0.369005   

   evenness_mean  evenness_sd  
0       0.410037     0.315279  
1       0.319149     0.244520  
2       0.332999     0.352048  


In [ ]:
final_df = project_summary.merge(df_per_project, on='project', how='left')

In [ ]:
# TODO investigate the identity of pollen across projects, rerun mods with the taxonomy 
# TODO shap feature importance

In [7]:
rf =MultiOutputRegressor(RandomForestRegressor(n_estimators = 200,
                          min_samples_split = 5,
                          min_samples_leaf = 1, 
                          max_features = 'log2', 
                          max_depth = None)) 

rf = rf.fit(X_train,y_train)
pred_train = rf.predict(X_train)
pred_test = rf.predict(X_test)

In [8]:
X_cols = np.load('../pollenGeolocation_saved/SI_X_cols.npy', allow_pickle=True)

In [ ]:
import shap
import numpy as np

# 1. Extract your MultiOutputRegressor
# Example:
# rf_model = rf_result["estimator"]

# 2. Loop over outputs (e.g., lat, lon)
shap_values_list = []

for i, estimator in enumerate(rf.estimators_):
    explainer = shap.TreeExplainer(estimator)
    shap_values = explainer.shap_values(X_test)
    shap_values_list.append(shap_values)

# 3. Average absolute SHAP values if you want a single importance score
mean_shap = np.mean(np.abs(np.stack(shap_values_list)), axis=0)

# 4. Wrap in DataFrame
shap_df = pd.DataFrame(mean_shap, columns=X_cols)
shap_df["project"] = project_test

# 5. Group by project
project_feature_importance = (
    shap_df.groupby("project")
    .agg(lambda x: np.mean(np.abs(x)))
    .transpose()
)

print(project_feature_importance)
